# TPAMI Revision Analysis — PE Strategies in ViT

This notebook runs the diagnostic analyses on already-trained ViT-Base
checkpoints for the TPAMI revision. It produces the per-layer mutual
information values, two-track structural diagnostics, linear-probe
metrics, and aggregate JSONs that feed Table 3, Table S1, and Figure 6
of the manuscript.

**Coverage** (controlled by `ANALYSE_IN100` / `ANALYSE_CIFAR` /
`ANALYSE_TINYIN` toggles at the top of the notebook):

- **ImageNet-100**: 5 PE types × 3 seeds (224×224, 16×16 patches, N=197)
- **CIFAR-100**: 5 PE types × 3 seeds (native 32×32, 4×4 patches,
  8×8 image grid + CLS, N=65)
- **TinyImageNet**: 4 PE types × seed 42 (224×224 upsampled, 16×16
  patches, N=197). 2D-ALiBi is a scope opt-out for the TinyImageNet
  diagnostic suite per Section 6.8 (Limitations).

**What it computes** for each model:

1. **Mutual information** — canonical discrete estimator
   $I(\text{query position}; \text{argmax target})$ per layer
2. **Two-track diagnostics** — embedding-space metrics (per-dimension
   entropy, variance, PCA) for additive PE (Learned, Sinusoidal);
   attention-space intrinsic-structural metrics (RoPE active dimensions
   and rotation wavelengths, ALiBi slope schedule, per-head intrinsic
   entropy) for attention-space PE (RoPE, 1D-ALiBi, 2D-ALiBi)
3. **Linear probes** — row + column probe with 5-fold stratified CV;
   applied only to additive PE
4. **Bootstrap CIs** — paired percentile bootstrap helper

**Methodology notes**:
- Per-dataset architecture matches actual checkpoint shapes
  (IN-100 and TinyImageNet share N=197; CIFAR-100 uses N=65)
- `with torch.no_grad()` wrapping throughout to avoid grad-tracking errors
- CIFAR-100 column-probe scores are reported separately from
  IN-100/TinyImageNet column probes (different patch-grid topology)
- Entropy histograms use $n_{\text{bins}}{=}64$, $\mathtt{density{=}true}$
  for consistency with the figures in `figures_revision.py`

**Datasets and prerequisites**:
- **ImageNet-100** val set: expected at `IN100_VAL_DIR` (Cell 2).
  Class subfolders. Must be pre-downloaded — not bootstrapped by this
  notebook.
- **CIFAR-100**: torchvision `datasets.CIFAR100(..., download=True)`
  bootstraps it automatically; nothing to pre-stage.
- **TinyImageNet**: a dedicated preparation cell (see "Prepare
  TinyImageNet validation set" below) downloads the dataset to Drive
  and reorganises the val split into the per-class layout that
  `ManualImageFolderDataset` expects. First-time preparation takes
  ~10-15 min (due to Drive write throughput for the ~10K val images);
  subsequent runs reuse the Drive cache. Skip by setting
  `ANALYSE_TINYIN = False`.

**Output**: one JSON per `(dataset, pe_type, seed)` plus per-dataset
`_summary.json` and `_aggregate.json` under
`OUTPUT_DIR/revision_results/{imagenet100, cifar100, tinyin}/`.
Existing JSONs are skipped on rerun for incremental work. The
`imagenet100/_aggregate.json` is the input to Figure 6 in
`figures_revision.py`.

**Hardware**: trained and analysed on NVIDIA RTX PRO 6000 Blackwell Server
Edition. Analysis is inference-only — any CUDA-capable GPU with ~6 GB
VRAM sufficient for a ViT-Base forward pass should work. The full
pass through 5 PE × 3 datasets completes in well under an hour
(actual wall time dominated by Drive I/O when reading checkpoints,
not compute).

## ⚙️ Configuration

**This is the only cell you typically need to edit.** Set paths to your
trained models and data here. Run-all from this point.


In [ ]:
# ============================================================
# CONFIG — edit these paths to match your Drive layout
# ============================================================

# --- Which datasets to analyse (set to False to skip a dataset) ---
ANALYSE_IMAGENET100 = True
ANALYSE_CIFAR100    = True
ANALYSE_TINYIN      = True       # NEW: TinyImageNet (single-seed, 4 PEs)

# --- Drive paths to trained model directories ---
MODELS_ROOT_IN100   = "/content/drive/MyDrive/Trained models_ImageNet100"
MODELS_ROOT_CIFAR   = "/content/drive/MyDrive/Trained models_CIFAR100"
MODELS_ROOT_TINYIN  = "/content/drive/MyDrive/Trained models_TinyImageNet"   # NEW

# --- Drive paths to validation data ---
IN100_VAL_DIR       = "/content/drive/MyDrive/imagenet100/val"
CIFAR_DATA_DIR      = "/content/drive/MyDrive/cifar100_data"
TINYIN_VAL_DIR      = "/content/drive/MyDrive/tiny-imagenet-200/val"        # NEW

# --- Output directory for JSON results ---
OUTPUT_DIR          = "/content/drive/MyDrive/revision_results"

# --- Which PE types and seeds to run (defaults for IN-100 / CIFAR) ---
PE_TYPES = ["learned", "sinusoidal", "rope", "alibi", "alibi_2d"]
SEEDS    = [42, 123, 456]

# --- TinyImageNet uses a reduced configuration (per paper Section 3.2) ---
# Single-seed evaluation, no 2D-ALiBi (scope opt-out per Limitations 6.8)
PE_TYPES_TINYIN = ["learned", "sinusoidal", "rope", "alibi"]
SEEDS_TINYIN    = [42]

# --- Analyses to run (set False to skip; useful for partial reruns) ---
RUN_MI                 = True   # Mutual Information per layer (Fix 1)
RUN_ALIBI_INTRINSIC    = True   # ALiBi bias tensor analysis (Fix 2a)
RUN_ROPE_INTRINSIC     = True   # RoPE rotation signature analysis (Fix 2b)
RUN_PROBE_V2           = True   # Row+col probe with proper CV (Fix 3)
RUN_DIMENSION_ENTROPY  = True   # Per-dim entropy (additive PE only)
RUN_DIMENSION_VARIANCE = True   # Per-dim variance (additive PE only)
RUN_NOISE_ABLATION     = False  # Slow (~5 min/model); rerun if needed
RUN_PE_REMOVAL         = False  # Subset of noise_ablation

# --- Compute settings ---
BATCH_SIZE  = 128
NUM_WORKERS = 4
N_BATCHES_MI = 50    # 50 * 128 = 6400 samples for MI estimation
DEVICE = "cuda"      # set to "cpu" for debugging without GPU

# --- Common model architecture (shared across datasets) ---
EMBED_DIM     = 768
DEPTH         = 12
NUM_HEADS     = 12
MLP_RATIO     = 4.0
DROPOUT       = 0.1

# --- Per-dataset architecture parameters ---
# IMPORTANT: CIFAR-100 models were trained on native 32x32 with 4x4 patches
# (giving 8x8 = 64 patches + 1 CLS = 65 positions), NOT upscaled to 224.
# This is the standard ViT-on-CIFAR setup and matches the checkpoints.
IN100_ARCH = {
    "img_size":             224,
    "patch_size":            16,
    "num_patches_per_side":  14,    # 14*14 = 196 patches
    "num_positions":        197,    # 196 + 1 CLS
    "num_classes":          100,
}
CIFAR_ARCH = {
    "img_size":              32,
    "patch_size":             4,
    "num_patches_per_side":   8,    # 8*8 = 64 patches
    "num_positions":         65,    # 64 + 1 CLS
    "num_classes":          100,
}
# TinyImageNet: native 64x64 upsampled to 224x224 (3.5x factor, see paper
# Section 3.2). Same patch grid as IN-100 (14x14, N=197) for direct
# comparability of representation-space diagnostics.
TINYIN_ARCH = {
    "img_size":             224,
    "patch_size":            16,
    "num_patches_per_side":  14,
    "num_positions":        197,
    "num_classes":          200,    # 200 classes (vs IN-100's 100)
}

print("Config loaded.")
print(f"  IN-100 models:    {MODELS_ROOT_IN100}")
print(f"    arch: {IN100_ARCH['img_size']}x{IN100_ARCH['img_size']} "
      f"img, {IN100_ARCH['patch_size']}x{IN100_ARCH['patch_size']} patches, "
      f"{IN100_ARCH['num_positions']} positions")
print(f"  CIFAR models:     {MODELS_ROOT_CIFAR}")
print(f"    arch: {CIFAR_ARCH['img_size']}x{CIFAR_ARCH['img_size']} "
      f"img, {CIFAR_ARCH['patch_size']}x{CIFAR_ARCH['patch_size']} patches, "
      f"{CIFAR_ARCH['num_positions']} positions")
print(f"  TinyIN models:    {MODELS_ROOT_TINYIN}")
print(f"    arch: {TINYIN_ARCH['img_size']}x{TINYIN_ARCH['img_size']} "
      f"img (upsampled from 64), {TINYIN_ARCH['patch_size']}x{TINYIN_ARCH['patch_size']} patches, "
      f"{TINYIN_ARCH['num_positions']} positions, {TINYIN_ARCH['num_classes']} classes")
print(f"  Output:           {OUTPUT_DIR}")
print(f"  PE types (IN-100 / CIFAR): {PE_TYPES}")
print(f"  PE types (TinyIN):         {PE_TYPES_TINYIN}")
print(f"  Seeds (IN-100 / CIFAR):    {SEEDS}")
print(f"  Seeds (TinyIN):            {SEEDS_TINYIN}")


Config loaded.
  IN-100 models:    /content/drive/MyDrive/Trained models_ImageNet100
    arch: 224x224 img, 16x16 patches, 197 positions
  CIFAR models:     /content/drive/MyDrive/Trained models_CIFAR100
    arch: 32x32 img, 4x4 patches, 65 positions
  TinyIN models:    /content/drive/MyDrive/Trained models_TinyImageNet
    arch: 224x224 img (upsampled from 64), 16x16 patches, 197 positions, 200 classes
  Output:           /content/drive/MyDrive/revision_results
  PE types (IN-100 / CIFAR): ['learned', 'sinusoidal', 'rope', 'alibi', 'alibi_2d']
  PE types (TinyIN):         ['learned', 'sinusoidal', 'rope', 'alibi']
  Seeds (IN-100 / CIFAR):    [42, 123, 456]
  Seeds (TinyIN):            [42]


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%run /content/drive/MyDrive/pe_experiment/00_setup_imagenet.py \
      --tar_path "/content/drive/MyDrive/pe_experiment/imagenet/ILSVRC2012_img_val.tar" \
      --labels_path "/content/drive/MyDrive/pe_experiment/val_labels.txt" \
      --classes_path "/content/drive/MyDrive/pe_experiment/imagenet100_classes.txt" \
      --output_dir "/content/imagenet100"

ImageNet-100 Setup
 Found: ILSVRC2012_img_val.tar
 Found: imagenet100_classes.txt
 Found: val_labels.txt

ImageNet-100 classes: 100
Val labels loaded: 50000 entries
Output directory: /content/imagenet100/val
Created 100 class folders

Extracting from: /content/drive/MyDrive/pe_experiment/imagenet/ILSVRC2012_img_val.tar
(This may take 5-10 minutes depending on Drive speed...)

Total images in tar: 50,000


Filtering: 100%|██████████| 50000/50000 [00:10<00:00, 4787.24img/s] 


Extraction complete!
  Images extracted: 5,000
  Images skipped:   45,000
  Expected:         5,000

 All 5,000 images extracted successfully.

Per-class image count (first 5):
  n01558993: 50 images
  n01692333: 50 images
  n01729322: 50 images
  n01735189: 50 images
  n01749939: 50 images

  Min images per class: 50
  Max images per class: 50
All classes have exactly 50 images.

Dataset ready at: /content/imagenet100/val


In [ ]:
import os
from torchvision import datasets

VAL = "/content/imagenet100/val"
print(f"Klasa: {len(os.listdir(VAL))}")
print(f"Prvih 3 ima slika:")
for c in sorted(os.listdir(VAL))[:3]:
    print(f"  {c}: {len(os.listdir(os.path.join(VAL, c)))} slika")

# Test ImageFolder ODMAH
ds = datasets.ImageFolder(VAL)
print(f"\nImageFolder USPEO: {len(ds)} slika u {len(ds.classes)} klasa")

Klasa: 100
Prvih 3 ima slika:
  n01558993: 50 slika
  n01692333: 50 slika
  n01729322: 50 slika

ImageFolder USPEO: 5000 slika u 100 klasa


## 2. Install dependencies

In [ ]:
!pip install -q timm scikit-learn scipy numpy matplotlib tqdm
print("Dependencies installed.")


Dependencies installed.


## 3. Imports and GPU check

In [ ]:
import os
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from scipy.stats import entropy as sp_entropy
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA: {torch.version.cuda}")


Using device: cuda
  GPU: Tesla T4
  CUDA: 12.8


## 4. Model architecture

These classes mirror `full_scale_experiment.py` exactly. Required to
load the saved checkpoints. Do not modify.


In [ ]:
# --- Patch embedding ---
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)                       # (B, embed_dim, H', W')
        x = x.flatten(2).transpose(1, 2)       # (B, N, embed_dim)
        return x


# --- Learned Positional Encoding ---
class LearnedPE(nn.Module):
    def __init__(self, num_positions, embed_dim):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.zeros(1, num_positions, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        return x + self.pos_embed


# --- Sinusoidal Positional Encoding ---
class SinusoidalPE(nn.Module):
    def __init__(self, num_positions, embed_dim):
        super().__init__()
        pe = torch.zeros(num_positions, embed_dim)
        position = torch.arange(0, num_positions, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


In [ ]:
# --- Rotary Position Embedding ---
class RoPE(nn.Module):
    def __init__(self, num_positions, head_dim):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer('inv_freq', inv_freq)
        t = torch.arange(num_positions, dtype=torch.float)
        freqs = torch.einsum('i,j->ij', t, inv_freq)
        self.register_buffer('cos_cached', freqs.cos().unsqueeze(0).unsqueeze(0))
        self.register_buffer('sin_cached', freqs.sin().unsqueeze(0).unsqueeze(0))

    def _rotate_half(self, x):
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat((-x2, x1), dim=-1)

    def forward(self, q, k, seq_len):
        cos = self.cos_cached[:, :, :seq_len, :]
        sin = self.sin_cached[:, :, :seq_len, :]
        cos = torch.cat([cos, cos], dim=-1)
        sin = torch.cat([sin, sin], dim=-1)
        q_rot = q * cos + self._rotate_half(q) * sin
        k_rot = k * cos + self._rotate_half(k) * sin
        return q_rot, k_rot


# --- ALiBi (1D) ---
class ALiBi(nn.Module):
    def __init__(self, num_heads, num_positions):
        super().__init__()
        ratio = 2 ** (-8.0 / num_heads)
        slopes = torch.tensor([ratio ** i for i in range(1, num_heads + 1)])
        self.register_buffer('slopes', slopes.view(1, num_heads, 1, 1))
        positions = torch.arange(num_positions)
        rel_dist = positions.unsqueeze(0) - positions.unsqueeze(1)
        rel_dist = rel_dist.abs().float()
        self.register_buffer('rel_dist', rel_dist.unsqueeze(0).unsqueeze(0))

    def get_bias(self, seq_len):
        return -self.slopes * self.rel_dist[:, :, :seq_len, :seq_len]


# --- 2D-ALiBi (Euclidean distance over patch grid) ---
class TwoDALiBi(nn.Module):
    def __init__(self, num_heads, num_patches_per_side=14, include_cls=True):
        super().__init__()
        self.num_heads = num_heads
        self.num_patches_per_side = num_patches_per_side
        self.include_cls = include_cls
        num_patches = num_patches_per_side ** 2
        num_positions = num_patches + (1 if include_cls else 0)
        ratio = 2 ** (-8.0 / num_heads)
        slopes = torch.tensor([ratio ** i for i in range(1, num_heads + 1)])
        self.register_buffer('slopes', slopes.view(1, num_heads, 1, 1))
        dist = torch.zeros(num_positions, num_positions)
        patch_offset = 1 if include_cls else 0
        coords = torch.zeros(num_patches, 2)
        for i in range(num_patches):
            coords[i, 0] = i // num_patches_per_side
            coords[i, 1] = i %  num_patches_per_side
        for i in range(num_patches):
            for j in range(num_patches):
                dr = coords[i, 0] - coords[j, 0]
                dc = coords[i, 1] - coords[j, 1]
                dist[i + patch_offset, j + patch_offset] = torch.sqrt(dr*dr + dc*dc)
        # Expose under both `dist_2d` and `rel_dist` for analysis compatibility
        self.register_buffer('dist_2d', dist.unsqueeze(0).unsqueeze(0))
        self.register_buffer('rel_dist', dist.unsqueeze(0).unsqueeze(0))

    def get_bias(self, seq_len):
        return -self.slopes * self.dist_2d[:, :, :seq_len, :seq_len]


In [ ]:
# --- Multi-Head Attention ---
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, pe_type='learned', num_positions=197):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.pe_type = pe_type
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        if pe_type == 'rope':
            self.rope = RoPE(num_positions, self.head_dim)
        elif pe_type == 'alibi':
            self.alibi = ALiBi(num_heads, num_positions)
        elif pe_type == 'alibi_2d':
            num_patches = num_positions - 1
            side = int(round(num_patches ** 0.5))
            assert side * side == num_patches, \
                f"alibi_2d requires square patch grid, got num_patches={num_patches}"
            self.alibi = TwoDALiBi(num_heads, num_patches_per_side=side,
                                    include_cls=True)

    def forward(self, x, return_attention=False):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        if self.pe_type == 'rope':
            q, k = self.rope(q, k, N)
        attn_bias = self.alibi.get_bias(N) if self.pe_type in ('alibi', 'alibi_2d') else None
        if not return_attention:
            x = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                                                dropout_p=0.1 if self.training else 0.0)
            attn_weights = None
        else:
            attn = (q @ k.transpose(-2, -1)) * self.scale
            if self.pe_type in ('alibi', 'alibi_2d'):
                attn = attn + attn_bias
            attn_weights = attn.softmax(dim=-1)
            x = (attn_weights @ v)
        x = x.transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        if return_attention:
            return x, attn_weights
        return x


# --- Transformer Block ---
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1,
                 pe_type='learned', num_positions=197):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, pe_type, num_positions)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, return_attention=False):
        if return_attention:
            attn_out, attn_weights = self.attn(self.norm1(x), return_attention=True)
            x = x + attn_out
            x = x + self.mlp(self.norm2(x))
            return x, attn_weights
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


In [ ]:
# --- Vision Transformer ---
class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0,
                 dropout=0.1, num_classes=100, pe_type='learned'):
        super().__init__()
        self.embed_dim = embed_dim
        self.depth = depth
        self.pe_type = pe_type
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        num_positions = self.patch_embed.num_patches + 1

        # Only additive PE methods get a pos_encoding module
        if pe_type == 'learned':
            self.pos_encoding = LearnedPE(num_positions, embed_dim)
        elif pe_type == 'sinusoidal':
            self.pos_encoding = SinusoidalPE(num_positions, embed_dim)
        else:
            self.pos_encoding = None  # RoPE / ALiBi apply within attention

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout,
                             pe_type, num_positions) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        if self.pos_encoding is not None:
            x = self.pos_encoding(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])

    def forward_with_attention(self, x):
        B = x.size(0)
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        if self.pos_encoding is not None:
            x = self.pos_encoding(x)
        attentions = []
        for block in self.blocks:
            x, attn = block(x, return_attention=True)
            attentions.append(attn)
        x = self.norm(x)
        return self.head(x[:, 0]), attentions


def build_model(pe_type, arch):
    """Construct a fresh ViT matching the given dataset's architecture.

    arch: dict with keys img_size, patch_size, num_patches_per_side,
          num_positions, num_classes.
    """
    return VisionTransformer(
        img_size=arch["img_size"],
        patch_size=arch["patch_size"],
        in_channels=3,
        embed_dim=EMBED_DIM, depth=DEPTH, num_heads=NUM_HEADS,
        mlp_ratio=MLP_RATIO, dropout=DROPOUT,
        num_classes=arch["num_classes"], pe_type=pe_type,
    )

print("Model architecture defined.")


Model architecture defined.


## 5. Revision-fix functions

Proper MI, two-track analyses, clean probes, bootstrap CI.
See `revision_fixes.py` for the standalone module with full docstrings.


In [ ]:
# ====== Fix 1: Proper MI estimator =========================
def _mutual_information_discrete(x, y, n_x_bins, n_y_bins):
    joint, _, _ = np.histogram2d(x, y, bins=[n_x_bins, n_y_bins],
                                  range=[[0, n_x_bins], [0, n_y_bins]])
    joint = joint.astype(np.float64)
    total = joint.sum()
    if total == 0:
        return 0.0
    pxy = joint / total
    px  = pxy.sum(axis=1, keepdims=True)
    py  = pxy.sum(axis=0, keepdims=True)
    nonzero = pxy > 0
    denom = px @ py
    ratio = np.zeros_like(pxy)
    ratio[nonzero] = pxy[nonzero] / denom[nonzero]
    mi = np.zeros_like(pxy)
    mi[nonzero] = pxy[nonzero] * np.log2(ratio[nonzero])
    return float(mi.sum())


@torch.no_grad()
def compute_mi_per_layer_v2(model, val_loader, device, n_batches=50):
    """I(query_position; argmax_target) per layer, in bits."""
    model.eval()
    num_layers = model.depth
    argmax_per_layer = [[] for _ in range(num_layers)]

    for batch_idx, (images, _) in enumerate(val_loader):
        if batch_idx >= n_batches:
            break
        images = images.to(device)
        _, attentions = model.forward_with_attention(images)
        for layer_idx, attn in enumerate(attentions):
            avg_attn = attn.mean(dim=1)
            argmax = avg_attn.argmax(dim=-1)
            argmax_per_layer[layer_idx].append(argmax.cpu().numpy())

    mi_values = []
    for layer_idx in range(num_layers):
        all_argmax = np.concatenate(argmax_per_layer[layer_idx], axis=0)
        total_B, N = all_argmax.shape
        positions = np.tile(np.arange(N), total_B)
        targets   = all_argmax.flatten()
        mi = _mutual_information_discrete(positions, targets, N, N)
        mi_values.append(mi)
    return mi_values


In [ ]:
# ====== Fix 2a: ALiBi intrinsic analysis ===================
def analyze_alibi_intrinsic(model):
    alibi = model.blocks[0].attn.alibi
    num_heads = alibi.slopes.shape[1]
    slopes = alibi.slopes.squeeze().cpu().numpy()
    rel_dist = alibi.rel_dist.squeeze().cpu().numpy()
    N = rel_dist.shape[0]

    bias_tensor = np.zeros((num_heads, N, N))
    for h in range(num_heads):
        bias_tensor[h] = -slopes[h] * rel_dist

    stacked = bias_tensor.reshape(num_heads * N, N)
    intrinsic_rank = int(np.linalg.matrix_rank(stacked, tol=1e-6))

    n_bins = 64
    per_head_entropy = np.zeros(num_heads)
    for h in range(num_heads):
        vals = bias_tensor[h].flatten()
        hist, _ = np.histogram(vals, bins=n_bins, density=True)
        hist = hist[hist > 0]; hist = hist / hist.sum()
        per_head_entropy[h] = sp_entropy(hist, base=2)

    effective_rf = np.where(np.abs(slopes) > 0, 1.0 / np.abs(slopes), np.inf)

    return {
        'intrinsic_rank':            intrinsic_rank,
        'per_head_entropy':          per_head_entropy.tolist(),
        'mean_entropy':              float(per_head_entropy.mean()),
        'slopes':                    slopes.tolist(),
        'effective_receptive_field': effective_rf.tolist(),
    }


# ====== Fix 2b: RoPE intrinsic analysis ====================
def analyze_rope_intrinsic(model):
    rope = model.blocks[0].attn.rope
    inv_freq = rope.inv_freq.cpu().numpy()
    N = rope.cos_cached.shape[2]
    wavelengths = 2 * np.pi / inv_freq
    active_fraction = float((wavelengths < N).mean())
    cos_cached = rope.cos_cached.squeeze().cpu().numpy()
    sin_cached = rope.sin_cached.squeeze().cpu().numpy()
    rotation_sig = np.concatenate([cos_cached, sin_cached], axis=1)
    intrinsic_rank = int(np.linalg.matrix_rank(rotation_sig, tol=1e-6))
    return {
        'theta_frequencies':     inv_freq.tolist(),
        'effective_wavelengths': wavelengths.tolist(),
        'active_dim_fraction':   active_fraction,
        'intrinsic_rank':        intrinsic_rank,
    }


In [ ]:
# ====== Fix 3: Clean probe analysis (row + column only) ====
def probe_analysis_v2(pe_matrix, num_patches_per_side=14, random_state=42):
    patches = pe_matrix[1:]  # drop CLS
    positions = np.arange(patches.shape[0])
    rows = positions // num_patches_per_side
    cols = positions %  num_patches_per_side

    results = {}
    for task_name, labels in [('row', rows), ('column', cols)]:
        clf = LogisticRegression(max_iter=2000, C=1.0)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
        scores = cross_val_score(clf, patches, labels, cv=cv, scoring='accuracy')
        results[task_name] = {
            'mean':     float(scores.mean() * 100),
            'std':      float(scores.std()  * 100),
            'per_fold': [float(s * 100) for s in scores],
        }
    return results


# ====== Bonus: Bootstrap CI for accuracy comparisons ========
def paired_bootstrap_ci(values_a, values_b, n_bootstrap=10000,
                         confidence=0.95, random_state=42):
    rng = np.random.default_rng(random_state)
    a = np.asarray(values_a, dtype=np.float64)
    b = np.asarray(values_b, dtype=np.float64)
    diffs = a - b
    n = len(a)
    boot_means = np.array([diffs[rng.integers(0, n, size=n)].mean()
                           for _ in range(n_bootstrap)])
    mean_diff = float(diffs.mean())
    alpha = 1 - confidence
    ci_low  = float(np.percentile(boot_means, 100 * alpha / 2))
    ci_high = float(np.percentile(boot_means, 100 * (1 - alpha / 2)))
    p_one = float((boot_means <= 0).mean()) if mean_diff >= 0 else             float((boot_means >= 0).mean())
    return {'mean_diff': mean_diff, 'ci_low': ci_low, 'ci_high': ci_high,
            'p_value':   2 * p_one}


# ====== Existing diagnostics (legacy support) ===============
def compute_dimension_entropy(pe_matrix, n_bins=64):
    entropies = []
    for d in range(pe_matrix.shape[1]):
        hist, _ = np.histogram(pe_matrix[:, d], bins=n_bins, density=True)
        hist = hist[hist > 0]; hist = hist / hist.sum()
        entropies.append(sp_entropy(hist, base=2))
    return np.array(entropies)


def compute_dimension_variance(pe_matrix):
    return np.var(pe_matrix[1:], axis=0)


def extract_additive_pe(model, pe_type):
    """Extract the additive PE matrix from Learned or Sinusoidal model."""
    if pe_type == 'learned':
        return model.pos_encoding.pos_embed.squeeze(0).cpu().numpy()
    elif pe_type == 'sinusoidal':
        return model.pos_encoding.pe.squeeze(0).cpu().numpy()
    else:
        return None  # RoPE / ALiBi handled via intrinsic analyses

print("Revision functions loaded.")


Revision functions loaded.


## 6. Checkpoint loader

Handles `torch.compile()`'s `_orig_mod.` prefix from training and
both `best_model.pth` and bare `best_model` filenames.


In [ ]:
def find_checkpoint(folder):
    """Find best_model checkpoint, with or without .pth extension."""
    for candidate in ['best_model.pth', 'best_model.pt', 'best_model']:
        p = os.path.join(folder, candidate)
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(f"No best_model[.pth|.pt|] found in {folder}")


def load_model(pe_type, seed, models_root, arch):
    """Load a trained model checkpoint using the dataset's architecture."""
    folder = os.path.join(models_root, f"{pe_type}_seed{seed}")
    ckpt_path = find_checkpoint(folder)

    model = build_model(pe_type, arch).to(device)
    state = torch.load(ckpt_path, map_location=device)
    state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
    if 'model' in state and isinstance(state['model'], dict):
        state = state['model']
        state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
    model.load_state_dict(state, strict=True)
    model.eval()
    return model


print("Checkpoint loader ready.")


Checkpoint loader ready.


## 7. Data loaders

ImageNet-100 uses `ImageFolder` on the validation split. CIFAR-100 is
downloaded by torchvision and resized to 224×224 to match training.


## Prepare TinyImageNet validation set

`get_tinyin_val_loader` (Cell 23) uses `ManualImageFolderDataset`, which
expects per-class subfolders. TinyImageNet's stock release ships the val
set as a flat folder of images with annotations in `val_annotations.txt`.
This cell ensures the dataset is prepared on Drive in the layout used by
`tinyimagenet_experiment.py`, and re-points `TINYIN_VAL_DIR` accordingly.

**Behaviour** (idempotent — re-run cheaply after first preparation):

1. If `MyDrive/tinyimagenet/tiny-imagenet-200/val_reorganised/` already
   exists (with 200 class folders), nothing happens. The path is reused.
2. Otherwise download `tiny-imagenet-200.zip` (~250 MB) from the Stanford
   CS231n mirror to `MyDrive/tinyimagenet/`, extract, and reorganise
   `val/images/*.JPEG` into `val_reorganised/<class>/<img>.JPEG`. First
   preparation takes ~10-15 min due to Drive write throughput; subsequent
   runs are instant.
3. Re-point `TINYIN_VAL_DIR` to the prepared `val_reorganised/` path so
   that Cell 23 picks it up automatically.

This matches the dataset preparation in `tinyimagenet_experiment.py`
(used during TinyImageNet training), so a single Drive cache serves
both training and analysis.

In [ ]:
import os
import urllib.request
import zipfile

TINYIN_DATA_ROOT = "/content/drive/MyDrive/tinyimagenet"
TINYIN_URL = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"


def _has_200_class_folders(path):
    """Return True iff `path` exists and contains 200 per-class subfolders."""
    if not os.path.isdir(path):
        return False
    entries = [e for e in os.scandir(path)
               if e.is_dir() and not e.name.startswith('.')]
    return len(entries) == 200


def prepare_tinyin_val():
    """Ensure TIN dataset is prepared at the canonical Drive location and
    point TINYIN_VAL_DIR at the reorganised val split."""
    global TINYIN_VAL_DIR

    extracted = os.path.join(TINYIN_DATA_ROOT, "tiny-imagenet-200")
    val_reorg = os.path.join(extracted, "val_reorganised")

    # Fast path: already prepared.
    if _has_200_class_folders(val_reorg):
        print(f"[OK] {val_reorg} already prepared (200 class folders).")
        TINYIN_VAL_DIR = val_reorg
        print(f"     TINYIN_VAL_DIR updated to: {TINYIN_VAL_DIR}")
        return

    # Slow path: prepare from scratch.
    os.makedirs(TINYIN_DATA_ROOT, exist_ok=True)

    # 1. Download (skip if zip already present)
    zip_path = os.path.join(TINYIN_DATA_ROOT, "tiny-imagenet-200.zip")
    if not os.path.isfile(zip_path) and not os.path.isdir(extracted):
        print(f"  Downloading TinyImageNet (~250 MB) to {zip_path}...")
        urllib.request.urlretrieve(TINYIN_URL, zip_path)
        print(f"  Download complete.")

    # 2. Extract (skip if already extracted)
    if not os.path.isdir(extracted):
        print(f"  Extracting {zip_path} to {TINYIN_DATA_ROOT}...")
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(TINYIN_DATA_ROOT)
        print(f"  Extraction complete.")

    # 3. Reorganise val/images/* into val_reorganised/<class>/*
    if not _has_200_class_folders(val_reorg):
        print(f"  Reorganising val/ into ImageFolder layout at "
              f"{val_reorg}...")
        val_dir = os.path.join(extracted, 'val')
        val_annotations = os.path.join(val_dir, 'val_annotations.txt')
        val_images_dir = os.path.join(val_dir, 'images')

        if not os.path.isfile(val_annotations):
            raise FileNotFoundError(
                f"val_annotations.txt missing from {val_dir}. "
                f"Extraction may have failed."
            )

        os.makedirs(val_reorg, exist_ok=True)
        n_linked = 0
        with open(val_annotations) as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) < 2:
                    continue
                fname, cls = parts[0], parts[1]
                cls_dir = os.path.join(val_reorg, cls)
                os.makedirs(cls_dir, exist_ok=True)
                src = os.path.join(val_images_dir, fname)
                dst = os.path.join(cls_dir, fname)
                if os.path.isfile(src) and not os.path.isfile(dst):
                    # Hardlinks are not supported on Drive; fall back to
                    # copy if needed.
                    try:
                        os.link(src, dst)
                    except OSError:
                        import shutil
                        shutil.copy2(src, dst)
                    n_linked += 1
        print(f"  Reorganised {n_linked} images into 200 class folders.")

    # 4. Sanity check.
    n_val_classes = len([d for d in os.listdir(val_reorg)
                          if os.path.isdir(os.path.join(val_reorg, d))])
    if n_val_classes != 200:
        raise RuntimeError(
            f"Expected 200 val classes, found {n_val_classes} in {val_reorg}"
        )

    # 5. Re-point the loader-expected variable.
    TINYIN_VAL_DIR = val_reorg
    print(f"[OK] TINYIN_VAL_DIR updated to: {TINYIN_VAL_DIR}")


if ANALYSE_TINYIN:
    prepare_tinyin_val()
else:
    print("[skip] ANALYSE_TINYIN is False; not preparing TinyImageNet val.")

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class ManualImageFolderDataset(Dataset):
    """Manual ImageFolder replacement — bypasses torchvision quirks."""

    IMG_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp')

    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform

        # Alphabetical class list (matches ImageFolder convention)
        self.classes = sorted(d.name for d in os.scandir(root) if d.is_dir())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        # Build (path, class_idx) samples manually
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root, class_name)
            class_idx = self.class_to_idx[class_name]
            for entry in os.scandir(class_dir):
                if entry.is_file() and entry.name.lower().endswith(self.IMG_EXTENSIONS):
                    self.samples.append((entry.path, class_idx))

        if len(self.samples) == 0:
            raise RuntimeError(
                f"No images found under {root}. "
                f"Found {len(self.classes)} class folders but no image files."
            )

        print(f"  ManualImageFolderDataset: {len(self.samples)} images, "
              f"{len(self.classes)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, target = self.samples[index]
        img = Image.open(path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, target


def get_in100_val_loader(val_dir, batch_size, num_workers):
    """Validation loader for ImageNet-100 — uses manual Dataset."""
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])
    ds = ManualImageFolderDataset(val_dir, transform=transform)
    return DataLoader(ds, batch_size=batch_size, shuffle=False,
                       num_workers=num_workers, pin_memory=True)


def get_cifar100_val_loader(data_dir, batch_size, num_workers):
    """Validation loader for CIFAR-100 at NATIVE 32x32 resolution.

    No upscale - matches the architecture the CIFAR-100 checkpoints
    were trained on (4x4 patches, 8x8 grid, 65 positions). Mean/std
    values match the original training script exactly.
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5071, 0.4867, 0.4408],
                              std=[0.2675, 0.2565, 0.2761]),
    ])
    ds = datasets.CIFAR100(root=data_dir, train=False, download=True,
                            transform=transform)
    return DataLoader(ds, batch_size=batch_size, shuffle=False,
                       num_workers=num_workers, pin_memory=True)


def get_tinyin_val_loader(val_dir, batch_size, num_workers):
    """Validation loader for TinyImageNet — upsample 64x64 -> 224x224.

    Matches the training-time pipeline per paper Section 3.2:
    native 64x64 images upsampled to 224x224 (3.5x factor) for direct
    architectural comparability with ImageNet-100 (14x14 patch grid,
    N=197). ImageNet mean/std normalisation, as is standard for
    TinyIN-on-ViT.

    NOTE on validation directory layout: TinyImageNet's stock release
    ships the val set as a flat folder of images with annotations in
    `val_annotations.txt`. ManualImageFolderDataset expects a per-class
    subfolder structure. If your val_dir does not have class subfolders,
    run the standard reorg script first (or train-time validation set).
    """
    transform = transforms.Compose([
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])
    ds = ManualImageFolderDataset(val_dir, transform=transform)
    return DataLoader(ds, batch_size=batch_size, shuffle=False,
                       num_workers=num_workers, pin_memory=True)


print("Data loader factories defined.")


Data loader factories defined.


## 8. Main analysis loop

Runs all configured analyses on every (dataset, pe_type, seed) and writes
one JSON file per model under `OUTPUT_DIR/{dataset}/{pe_type}_seed{seed}.json`.

You can interrupt and resume — the loop skips runs whose JSON already exists.


In [ ]:
def run_analyses_for_model(model, pe_type, val_loader, num_patches_per_side):
    """Run all enabled analyses on a single model. Returns a dict."""
    results = {'pe_type': pe_type}

    # Disable gradient tracking for the entire analysis pass
    with torch.no_grad():

        # Track 1: embedding-space analyses (additive PE only)
        pe_matrix = extract_additive_pe(model, pe_type)
        if pe_matrix is not None:
            if hasattr(pe_matrix, 'detach'):
                pe_matrix = pe_matrix.detach().cpu().numpy()

            if RUN_DIMENSION_ENTROPY:
                ent = compute_dimension_entropy(pe_matrix)
                results['dimension_entropy'] = {
                    'per_dim': ent.tolist(),
                    'mean':    float(ent.mean()),
                    'std':     float(ent.std()),
                }
            if RUN_DIMENSION_VARIANCE:
                var = compute_dimension_variance(pe_matrix)
                results['dimension_variance'] = {
                    'per_dim':         var.tolist(),
                    'mean':            float(var.mean()),
                    'std':             float(var.std()),
                    'fraction_active': float((var > 1e-4).mean()),
                }
            if RUN_PROBE_V2:
                results['probe'] = probe_analysis_v2(
                    pe_matrix, num_patches_per_side=num_patches_per_side)
        else:
            results['dimension_entropy']  = None
            results['dimension_variance'] = None
            results['probe']              = None

        # Track 2: attention-space intrinsic analyses
        if pe_type in ('alibi', 'alibi_2d') and RUN_ALIBI_INTRINSIC:
            results['alibi_intrinsic'] = analyze_alibi_intrinsic(model)
        if pe_type == 'rope' and RUN_ROPE_INTRINSIC:
            results['rope_intrinsic'] = analyze_rope_intrinsic(model)

        # Track 3: MI (all PE types)
        if RUN_MI:
            mi_values = compute_mi_per_layer_v2(model, val_loader, device,
                                                 n_batches=N_BATCHES_MI)
            # mi_values is a plain Python list of floats from the helper,
            # but guard against tensor leakage just in case
            if isinstance(mi_values, torch.Tensor):
                mi_values = mi_values.detach().cpu().numpy().tolist()
            results['mi_per_layer'] = {
                'values': list(mi_values),
                'mean':   float(np.mean(mi_values)),
                'max':    float(np.max(mi_values)),
                'min':    float(np.min(mi_values)),
            }

    return results


def analyse_dataset(dataset_name, models_root, val_loader, arch,
                    pe_types=None, seeds=None):
    """Run analyses for all PE x seed combinations in one dataset.

    Args:
        pe_types: list of PE types to run; defaults to global PE_TYPES
        seeds:    list of seeds to run;    defaults to global SEEDS

    These overrides let TinyImageNet run a reduced grid (4 PEs, 1 seed)
    without changing the IN-100 / CIFAR runs.
    """
    if pe_types is None:
        pe_types = PE_TYPES
    if seeds is None:
        seeds = SEEDS

    out_dir = os.path.join(OUTPUT_DIR, dataset_name)
    os.makedirs(out_dir, exist_ok=True)
    all_results = {}
    num_patches_per_side = arch["num_patches_per_side"]

    for pe_type in pe_types:
        for seed in seeds:
            tag = f"{pe_type}_seed{seed}"
            json_path = os.path.join(out_dir, f"{tag}.json")

            if os.path.isfile(json_path):
                print(f"[{dataset_name}] {tag}  -- already done, loading from cache")
                with open(json_path) as f:
                    all_results[tag] = json.load(f)
                continue

            t0 = time.time()
            print(f"[{dataset_name}] {tag}  -- loading model ...", end=' ', flush=True)
            try:
                model = load_model(pe_type, seed, models_root, arch)
            except Exception as e:
                print(f"FAILED to load: {e}")
                all_results[tag] = {'error': str(e)}
                continue
            print(f"done. Running analyses ...", end=' ', flush=True)

            try:
                res = run_analyses_for_model(model, pe_type, val_loader,
                                              num_patches_per_side)
                res['runtime_seconds'] = time.time() - t0
                with open(json_path, 'w') as f:
                    json.dump(res, f, indent=2)
                all_results[tag] = res
                print(f"OK ({res['runtime_seconds']:.1f}s)")
            except Exception as e:
                print(f"ANALYSIS FAILED: {e}")
                all_results[tag] = {'error': str(e)}

            del model
            torch.cuda.empty_cache()

    summary_path = os.path.join(out_dir, "_summary.json")
    with open(summary_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"\nDataset summary written: {summary_path}")
    return all_results


# Run the analyses
all_results = {}

if ANALYSE_IMAGENET100:
    print("\n" + "="*60)
    print("ImageNet-100 analyses (224x224, 16x16 patches, 197 positions)")
    print("="*60)
    in100_val = get_in100_val_loader(IN100_VAL_DIR, BATCH_SIZE, NUM_WORKERS)
    all_results['imagenet100'] = analyse_dataset(
        'imagenet100', MODELS_ROOT_IN100, in100_val, IN100_ARCH,
    )

if ANALYSE_CIFAR100:
    print("\n" + "="*60)
    print("CIFAR-100 analyses (32x32 native, 4x4 patches, 65 positions)")
    print("="*60)
    cifar_val = get_cifar100_val_loader(CIFAR_DATA_DIR, BATCH_SIZE, NUM_WORKERS)
    all_results['cifar100'] = analyse_dataset(
        'cifar100', MODELS_ROOT_CIFAR, cifar_val, CIFAR_ARCH,
    )

if ANALYSE_TINYIN:
    print("\n" + "="*60)
    print("TinyImageNet analyses (224x224 upsampled, 16x16 patches, 197 positions)")
    print(f"  Single-seed evaluation: seeds={SEEDS_TINYIN}")
    print(f"  No 2D-ALiBi (scope opt-out): pe_types={PE_TYPES_TINYIN}")
    print("="*60)
    tinyin_val = get_tinyin_val_loader(TINYIN_VAL_DIR, BATCH_SIZE, NUM_WORKERS)
    all_results['tinyin'] = analyse_dataset(
        'tinyin', MODELS_ROOT_TINYIN, tinyin_val, TINYIN_ARCH,
        pe_types=PE_TYPES_TINYIN, seeds=SEEDS_TINYIN,
    )

master_path = os.path.join(OUTPUT_DIR, "_master_summary.json")
with open(master_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\n✅ Master summary written: {master_path}")



ImageNet-100 analyses (224x224, 16x16 patches, 197 positions)
  ManualImageFolderDataset: 1654 images, 100 classes
[imagenet100] learned_seed42  -- already done, loading from cache


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[imagenet100] learned_seed123  -- already done, loading from cache
[imagenet100] learned_seed456  -- already done, loading from cache
[imagenet100] sinusoidal_seed42  -- already done, loading from cache
[imagenet100] sinusoidal_seed123  -- already done, loading from cache
[imagenet100] sinusoidal_seed456  -- already done, loading from cache
[imagenet100] rope_seed42  -- already done, loading from cache
[imagenet100] rope_seed123  -- already done, loading from cache
[imagenet100] rope_seed456  -- already done, loading from cache
[imagenet100] alibi_seed42  -- already done, loading from cache
[imagenet100] alibi_seed123  -- already done, loading from cache
[imagenet100] alibi_seed456  -- already done, loading from cache
[imagenet100] alibi_2d_seed42  -- already done, loading from cache
[imagenet100] alibi_2d_seed123  -- already done, loading from cache
[imagenet100] alibi_2d_seed456  -- already done, loading from cache

Dataset summary written: /content/drive/MyDrive/revision_results/ima

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/tiny-imagenet-200/val'

## 9. Quick aggregate statistics

Aggregates results across seeds and computes bootstrap CIs for the main
accuracy comparisons. Useful for the Methods section's statistical claims.


In [ ]:
def aggregate_across_seeds(dataset_results):
    """Aggregate per-seed results into mean±std for each PE type."""
    agg = {}
    for pe_type in PE_TYPES:
        per_seed = []
        for seed in SEEDS:
            tag = f"{pe_type}_seed{seed}"
            if tag in dataset_results and 'error' not in dataset_results[tag]:
                per_seed.append(dataset_results[tag])
        if not per_seed:
            continue

        # MI per layer (aggregate across seeds)
        if 'mi_per_layer' in per_seed[0]:
            mi_arrays = np.array([r['mi_per_layer']['values'] for r in per_seed])
            agg.setdefault(pe_type, {})['mi_per_layer'] = {
                'mean_per_layer': mi_arrays.mean(axis=0).tolist(),
                'std_per_layer':  mi_arrays.std(axis=0).tolist(),
                'overall_mean':   float(mi_arrays.mean()),
                'overall_std':    float(mi_arrays.std()),
            }
        # Probe results
        if per_seed[0].get('probe'):
            for axis in ('row', 'column'):
                vals = [r['probe'][axis]['mean'] for r in per_seed]
                agg.setdefault(pe_type, {}).setdefault('probe', {})[axis] = {
                    'mean': float(np.mean(vals)),
                    'std':  float(np.std(vals)),
                }
        # Entropy / variance
        if per_seed[0].get('dimension_entropy'):
            ents = [r['dimension_entropy']['mean'] for r in per_seed]
            agg.setdefault(pe_type, {})['dimension_entropy_mean'] = {
                'mean': float(np.mean(ents)), 'std': float(np.std(ents)),
            }
        if per_seed[0].get('dimension_variance'):
            vars_ = [r['dimension_variance']['mean'] for r in per_seed]
            agg.setdefault(pe_type, {})['dimension_variance_mean'] = {
                'mean': float(np.mean(vars_)), 'std': float(np.std(vars_)),
            }
    return agg


# Aggregate and print summary
print("=" * 60)
print("AGGREGATE RESULTS")
print("=" * 60)

for dataset_name, results in all_results.items():
    print(f"\n[{dataset_name}]")
    agg = aggregate_across_seeds(results)

    for pe_type, metrics in agg.items():
        print(f"  {pe_type:12s}", end="")
        if 'mi_per_layer' in metrics:
            mi = metrics['mi_per_layer']
            print(f" | MI mean = {mi['overall_mean']:.3f} +/- {mi['overall_std']:.3f} bits", end="")
        if 'probe' in metrics:
            r = metrics['probe'].get('row', {})
            c = metrics['probe'].get('column', {})
            if r and c:
                print(f" | row={r['mean']:.1f}+/-{r['std']:.1f}%, "
                      f"col={c['mean']:.1f}+/-{c['std']:.1f}%", end="")
        print()

    # Save aggregate JSON
    agg_path = os.path.join(OUTPUT_DIR, dataset_name, "_aggregate.json")
    with open(agg_path, 'w') as f:
        json.dump(agg, f, indent=2)
    print(f"  -> aggregate written: {agg_path}")

print("\n✅ Done. JSON files ready for paper writing.")


AGGREGATE RESULTS

[imagenet100]
  learned      | MI mean = 1.197 +/- 1.442 bits | row=76.2+/-1.3%, col=31.3+/-5.1%
  sinusoidal   | MI mean = 1.751 +/- 1.904 bits | row=89.3+/-0.0%, col=0.0+/-0.0%
  rope         | MI mean = 2.627 +/- 1.932 bits
  alibi        | MI mean = 5.167 +/- 1.165 bits
  alibi_2d     | MI mean = 3.407 +/- 1.389 bits
  -> aggregate written: /content/drive/MyDrive/revision_results/imagenet100/_aggregate.json

[cifar100]
  learned      | MI mean = 0.880 +/- 1.149 bits | row=51.9+/-1.9%, col=58.7+/-0.7%
  sinusoidal   | MI mean = 1.201 +/- 1.470 bits | row=86.0+/-0.0%, col=0.0+/-0.0%
  rope         | MI mean = 1.855 +/- 1.666 bits
  alibi        | MI mean = 3.954 +/- 0.629 bits
  -> aggregate written: /content/drive/MyDrive/revision_results/cifar100/_aggregate.json

✅ Done. JSON files ready for paper writing.


In [ ]:
import json, os, statistics as st

OUT = "/content/drive/MyDrive/Trained models_ImageNet100"

print("2D-ALiBi accuracy rezultati:")
print("=" * 60)
accs = []
for seed in [42, 123, 456]:
    path = os.path.join(OUT, f"alibi_2d_seed{seed}", "training_history.json")
    if os.path.isfile(path):
        with open(path) as f:
            h = json.load(f)
        best = max(h['val_acc'])
        accs.append(best)
        print(f"  Seed {seed}: best val_acc = {best:.2f}%")

if accs:
    mean = st.mean(accs)
    std = st.stdev(accs) if len(accs) > 1 else 0
    print("-" * 60)
    print(f"  Mean: {mean:.2f}% ± {std:.2f}")
    print()
    print(f"  Compare:")
    print(f"    1D-ALiBi:   81.05% ± 0.28")
    print(f"    2D-ALiBi:   {mean:.2f}% ± {std:.2f}")
    print(f"    Diff:       {mean - 81.05:+.2f}pp")
    print()
    if mean > 82:
        print(f"  ✅ 2D-ALiBi BOLJI od 1D — topological mismatch potvrđeno")
    elif mean > 81:
        print(f"  ≈ 2D-ALiBi blago bolji — slabi efekat")
    else:
        print(f"  ⚠️ 2D-ALiBi NIJE bolji — neočekivano, treba interpretacija")

2D-ALiBi accuracy rezultati:
  Seed 42: best val_acc = 80.94%
  Seed 123: best val_acc = 81.64%
  Seed 456: best val_acc = 80.94%
------------------------------------------------------------
  Mean: 81.17% ± 0.40

  Compare:
    1D-ALiBi:   81.05% ± 0.28
    2D-ALiBi:   81.17% ± 0.40
    Diff:       +0.12pp

  ≈ 2D-ALiBi blago bolji — slabi efekat
